In [ ]:
import numpy as np
from collections import defaultdict

# -----------------------------
# 1. Sample corpus
# -----------------------------
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat chased the dog",
    "the dog barked loudly"
]

# -----------------------------
# 2. Build vocabulary
# -----------------------------
def build_vocab(corpus):
    vocab = {}
    for sentence in corpus:
        for word in sentence.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(corpus)
vocab_size = len(vocab)

# -----------------------------
# 3. Build co-occurrence matrix
# -----------------------------
def build_cooccurrence_matrix(corpus, vocab, window_size=2):
    matrix = np.zeros((len(vocab), len(vocab)))

    for sentence in corpus:
        words = sentence.split()
        for i, word in enumerate(words):
            word_idx = vocab[word]

            # Context window
            for j in range(max(0, i - window_size), min(len(words), i + window_size + 1)):
                if i != j:
                    context_word = words[j]
                    context_idx = vocab[context_word]
                    matrix[word_idx][context_idx] += 1

    return matrix

co_matrix = build_cooccurrence_matrix(corpus, vocab)

print(co_matrix)
print()

# -----------------------------
# 4. Apply SVD
# -----------------------------
U, S, Vt = np.linalg.svd(co_matrix)

# -----------------------------
# 5. Reduce dimensions
# -----------------------------
k = 3  # embedding size
U_k = U[:, :k]
S_k = np.diag(S[:k])

# Word embeddings
embeddings = np.dot(U_k, S_k)

# -----------------------------
# 6. Inspect embeddings
# -----------------------------
index_to_word = {i: w for w, i in vocab.items()}

for i in range(len(vocab)):
    print(f"{index_to_word[i]} → {embeddings[i]}")

[[0. 3. 4. 2. 1. 3. 1. 2. 1. 0.]
 [3. 0. 1. 1. 0. 0. 0. 1. 0. 0.]
 [4. 1. 0. 2. 0. 1. 0. 0. 0. 0.]
 [2. 1. 2. 0. 1. 1. 1. 0. 0. 0.]
 [1. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [3. 0. 1. 1. 0. 0. 0. 1. 1. 1.]
 [1. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [2. 1. 0. 0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 1. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 1. 0. 0. 1. 0.]]

the → [-5.25248126  4.13887057 -0.1836549 ]
cat → [-2.8293772  -1.670814   -0.85924435]
sat → [-3.79983117 -2.46732183  0.80610801]
on → [-2.99176992  0.27831881  0.52489652]
mat → [-0.95442165 -0.79013509 -0.1546858 ]
dog → [-2.99992838 -1.64793317 -1.23422813]
log → [-0.95442165 -0.79013509 -0.1546858 ]
chased → [-1.89098788 -0.88705165  1.1154796 ]
barked → [-1.00909614 -0.51477258  0.4897531 ]
loudly → [-0.46411732  0.38685905  0.33747265]


## Dataset

In [ ]:
!pip install -q langdetect
!pip install gensim
!gdown 18M7lJHV6Y4pJTsFzDw8_mf2_cQMuFcxE

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 22.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 59.1 MB/s eta 0:00:00
Downloading...
From (original): https://drive.google.com/uc?id=18M7lJHV6Y4pJTsFzDw8_mf2_cQMuFcxE
From (redirected): https://drive.google.com/uc?id=18M7lJHV6Y4pJTsFzDw8_mf2_cQMuFcxE&confirm=t&uuid=20cba226-8953-493a-92c2-bfb5e64d22b0
To: /content/metadata.csv
100% 1.65G/1.65G [00:20<00:00, 81.7MB/s]


In [ ]:
import spacy
import string
import warnings

import numpy as np
import pandas as pd

from pprint import pprint
from IPython.utils import io
from tqdm.notebook import tqdm
from gensim.models import Word2Vec
from langdetect import DetectorFactory, detect
from IPython.core.display import HTML, display
from IPython.display import Image
from spacy.lang.en.stop_words import STOP_WORDS

warnings.filterwarnings('ignore')
tqdm.pandas()

In [ ]:
DATA = pd.read_csv("metadata.csv").sample(100000) # taking only 100000 research papers
DATA.reset_index(inplace=True, drop=True)
print(DATA.columns)
DATA.head(2)

Index(['cord_uid', 'sha', 'source_x', 'title', 'doi', 'pmcid', 'pubmed_id',
       'license', 'abstract', 'publish_time', 'authors', 'journal', 'mag_id',
       'who_covidence_id', 'arxiv_id', 'pdf_json_files', 'pmc_json_files',
       'url', 's2_id'],
      dtype='object')


,cord_uid,sha,source_x,title,doi,pmcid,pubmed_id,license,abstract,publish_time,authors,journal,mag_id,who_covidence_id,arxiv_id,pdf_json_files,pmc_json_files,url,s2_id
0,48hsdymv,NaN,Medline,Estimates of the Potential Impact of the COVID...,10.1363/46e9020,NaN,32343244.0,unk,The novel coronavirus (SARS-CoV-2) that causes...,2020-04-16,"Riley, Taylor; Sully, Elizabeth; Ahmed, Zara; ...",International perspectives on sexual and repro...,NaN,NaN,NaN,NaN,NaN,https://doi.org/10.1363/46e9020; https://www.n...,216595145.0
1,udxqk1wn,NaN,WHO,The application of empathetic learning in faci...,NaN,NaN,NaN,unk,This study aims to be able to apply the empath...,2020,"Sarkadi, Casmana; A., R.",International Journal of Psychosocial Rehabili...,NaN,#831019,NaN,NaN,NaN,NaN,221292436.0


In [ ]:
df_covid = pd.DataFrame(columns=['paper_id', 'title','abstract', 'doi'])
df_covid['paper_id'] = DATA.sha
df_covid['title'] = DATA.title
df_covid['abstract'] = DATA.abstract
df_covid['doi'] = DATA.doi

df_covid.head()

,paper_id,title,abstract,doi
0,NaN,Estimates of the Potential Impact of the COVID...,The novel coronavirus (SARS-CoV-2) that causes...,10.1363/46e9020
1,NaN,The application of empathetic learning in faci...,This study aims to be able to apply the empath...,NaN
2,e9d30d0f47f7c6f582900284011e12bb3859b150,Use of aerosolised medications at home for COV...,NaN,10.1016/s2213-2600(20)30270-8
3,NaN,Emprego da reação em cadeia pela polimerase em...,A estirpe Fromm de Leptospira interrogans soro...,NaN
4,NaN,Characteristics of COVID-19 at a non-COVID ter...,The pandemic of COVID-19 has emerged as a seri...,NaN


#### Dropping duplicates

In [ ]:
df_covid.drop_duplicates(['abstract'], inplace=True)
df_covid.dropna(inplace=True)
df_covid.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29581 entries, 6 to 99999
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   paper_id  29581 non-null  object
 1   title     29581 non-null  object
 2   abstract  29581 non-null  object
 3   doi       29581 non-null  object
dtypes: object(4)
memory usage: 1.1+ MB


#### We will focus on only publications in English and drop anything that is non-English.

In [ ]:
# set seed
DetectorFactory.seed = 0

# hold label - language
languages = []

# loop through each text
for ii in tqdm(range(0,len(df_covid))):
    text = df_covid.iloc[ii]['abstract'].split(" ")

    lang = "en"
    try:
        if len(text) > 50:
            lang = detect(" ".join(text[:50]))
        elif len(text) > 0:
            lang = detect(" ".join(text[:len(text)]))
    except Exception as e:
        all_words = set(text)
        try:
            lang = detect(" ".join(all_words))

        except Exception as e:
            lang = "unknown"
            pass

    # Appending to language label
    languages.append(lang)

  0%|          | 0/29581 [00:00<?, ?it/s]

#### Let's look at the numbers of articles for each language.

In [ ]:
languages_dict = {}
for lang in set(languages):
    languages_dict[lang] = languages.count(lang)

print("Total: {}\n".format(len(languages)))
print(languages_dict)

Total: 29581

{'fr': 49, 'ro': 1, 'pt': 6, 'es': 69, 'unknown': 2, 'it': 7, 'de': 72, 'nl': 11, 'zh-cn': 1, 'en': 29363}


#### Since most of the publications are in English, we can safely drop the rest

In [ ]:
df_covid['language'] = languages
df_covid = df_covid[df_covid['language'] == 'en']
df_covid.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29363 entries, 6 to 99999
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   paper_id  29363 non-null  object
 1   title     29363 non-null  object
 2   abstract  29363 non-null  object
 3   doi       29363 non-null  object
 4   language  29363 non-null  object
dtypes: object(5)
memory usage: 1.3+ MB


#### We also do not need the language feature column.

In [ ]:
df_covid = df_covid.drop(['language'], axis = 1)
df_covid.head()

,paper_id,title,abstract,doi
6,1747b7993bf64ae621abd516fd7bba9ea19925b6,From chaos to control – experiences of healthc...,BACKGROUND: The Coronavirus disease 2019 (COVI...,10.1186/s12913-021-07248-9
10,7e43c066f525790941c3fbe6c2a5e497f90f7087; f676...,The Long-Term Effects of Labor Market Entry in...,This study investigates the long-term effects ...,10.1016/j.labeco.2020.101926
11,5aae92d00dbccb18ae848851027fbfc3d830a7f1,Emerging evidence of myocardial injury in COVI...,Although Coronavirus Disease 2019 (COVID-19) p...,10.7150/thno.50788
12,e1a02e6c693400bcf277fe450d96850bf8d9dcd9,Qualitative Analysis of Fractal-Fractional ord...,"In this manuscript, a qualitative analysis of ...",10.1016/j.aej.2020.09.020
19,b8f4aadb2118430ce4faa3bd24720fb1a6e90554,Ambulatory Rehabilitation of Patients Hospital...,The current COVID‐19 pandemic has rapidly affe...,10.1002/pmrj.12506


#### Use text pre-processing techniques to get rid of punctuations and stop words

In [ ]:
punctuations = string.punctuation
stopwords = list(STOP_WORDS)

#### Creating custom list of stop-words based on our corpus

In [ ]:
custom_stop_words = [
    'doi', 'preprint', 'copyright', 'peer', 'reviewed', 'org', 'https', 'et', 'al', 'author', 'figure',
    'rights', 'reserved', 'permission', 'used', 'using', 'biorxiv', 'medrxiv', 'license', 'fig', 'fig.',
    'al.', 'Elsevier', 'PMC', 'CZI', 'www'
]

for w in custom_stop_words:
    if w not in stopwords:
        stopwords.append(w)

#### Cleaning abstracts

In [ ]:
def pre_processor(sentence):
    mytokens = sentence.split(' ')
    mytokens = [word.lower() for word in mytokens if word not in stopwords and word not in punctuations]
    mytokens = " ".join([i for i in mytokens])
    return mytokens

df_covid["processed_abstract"] = df_covid["abstract"].progress_apply(pre_processor)
df_covid.head()

  0%|          | 0/29363 [00:00<?, ?it/s]

,paper_id,title,abstract,doi,processed_abstract
6,1747b7993bf64ae621abd516fd7bba9ea19925b6,From chaos to control – experiences of healthc...,BACKGROUND: The Coronavirus disease 2019 (COVI...,10.1186/s12913-021-07248-9,background: the coronavirus disease 2019 (covi...
10,7e43c066f525790941c3fbe6c2a5e497f90f7087; f676...,The Long-Term Effects of Labor Market Entry in...,This study investigates the long-term effects ...,10.1016/j.labeco.2020.101926,this study investigates long-term effects init...
11,5aae92d00dbccb18ae848851027fbfc3d830a7f1,Emerging evidence of myocardial injury in COVI...,Although Coronavirus Disease 2019 (COVID-19) p...,10.7150/thno.50788,although coronavirus disease 2019 (covid-19) p...
12,e1a02e6c693400bcf277fe450d96850bf8d9dcd9,Qualitative Analysis of Fractal-Fractional ord...,"In this manuscript, a qualitative analysis of ...",10.1016/j.aej.2020.09.020,"in manuscript, qualitative analysis mathematic..."
19,b8f4aadb2118430ce4faa3bd24720fb1a6e90554,Ambulatory Rehabilitation of Patients Hospital...,The current COVID‐19 pandemic has rapidly affe...,10.1002/pmrj.12506,the current covid‐19 pandemic rapidly affected...


In [ ]:
df_covid["processed_abstract"].iloc[0]

'background: the coronavirus disease 2019 (covid-19) pandemic caused overwhelming challenges healthcare systems worldwide. healthcare workers (hcws) faced particular challenges: exposed coronavirus sars-cov-2 caring patients having new potentially life-threatening disease. the aim study explore hcws swedish healthcare system perceived work situation phase covid-19 pandemic 2020. methods: focus group discussions interviews hcws performed june october 2020 swedish healthcare region. a purposeful sampling approach select variety professions (physicians, nurses, nurse aides cleaners) workplaces (hospital inpatient wards, emergency department, nursing home home care service). qualitative content analysis data analysis. results: in total, 51 hcws participated focus group discussions hcw participated individual interview. the content analysis identified main categories: ‘concerns risk infection transmission infection others’, ‘transition chaos managing new challenging work situation’. the fin

#### To train the Word2Vec model, we'll have to convert the sentences into list of words

In [ ]:
abstracts = df_covid['processed_abstract'].values

## Let us use Spacy for faster tokenization
nlp = spacy.load('en_core_web_sm', disable=["tagger", "ner"])
nlp.add_pipe('sentencizer')

def tokenize_sentences(sentence):
    sentence_corpus = []
    doc = nlp(sentence)
    sentences = [sent.text.strip() for sent in doc.sents]
    for sent in sentences:
        processed_sent_list = sent.split(" ")
        sentence_corpus.append(processed_sent_list)
    return sentence_corpus

df_covid['tokenized_abstract'] = df_covid['processed_abstract'].progress_apply(lambda x: tokenize_sentences(x))

corpus_data = df_covid['tokenized_abstract'].to_list()
word2vec_corpus = [item for items in corpus_data for item in items]

  0%|          | 0/29363 [00:00<?, ?it/s]

In [ ]:
word2vec_corpus[0]

['background:',
 'the',
 'coronavirus',
 'disease',
 '2019',
 '(covid-19)',
 'pandemic',
 'caused',
 'overwhelming',
 'challenges',
 'healthcare',
 'systems',
 'worldwide.']

In [ ]:
len(word2vec_corpus)

276222

## Lets use Skip-gram

#### Let us calculate the centroid for each abstract using the vectors of all the words incorporating the abstract.

In [ ]:
model = Word2Vec(
    word2vec_corpus,
    min_count=3, #Ignore rare words. Only include words which occur atleast 3 times.
    vector_size=100, # Embedding size
    workers=4, # No of threads for parallel processing in CPU.
    window=5, # Context window size.
    sg=1, # Skip-gram is True
    negative=5 # random K negative sample
)

## Example of embedings

In [ ]:
model.wv["king"].shape # (100,)

(100,)

In [ ]:
model.wv["king"]

array([-0.24526735,  0.3504632 ,  0.0936645 ,  0.27482983, -0.0301831 ,
        0.1120755 ,  0.18813038,  0.04216397,  0.44409937, -0.1638201 ,
       -0.6037048 ,  0.00638657, -0.07937442, -0.3523516 ,  0.01260829,
       -0.58475363, -0.09351096, -0.19096994,  0.00663879, -0.2627238 ,
        0.12196688, -0.11231445,  0.14257891,  0.06282611, -0.09979661,
       -0.20362721, -0.33880007,  0.42074677, -0.5511429 ,  0.06819965,
        0.27278346,  0.35301393,  0.12171335,  0.1778952 , -0.28564677,
       -0.1549415 ,  0.00588159,  0.49447274,  0.01745186, -0.38732043,
        0.17510061, -0.2365024 , -0.12736572, -0.14339608,  0.11197416,
       -0.03815982, -0.17347427,  0.08106418,  0.00347989,  0.11750196,
        0.21179911, -0.6299582 , -0.81632394, -0.48657444, -0.596853  ,
       -0.3426048 , -0.26970112, -0.09101865, -0.42006928,  0.31798193,
        0.00635871,  0.44059262, -0.13108939, -0.1985178 , -1.0132186 ,
        0.23773617, -0.48112488, -0.0502362 , -0.6691995 ,  0.19

In [ ]:
model.wv["coronavirus"]

array([ 0.531666  , -0.49637324,  0.29478237,  0.6448489 ,  0.08439831,
       -0.60924906,  0.44305137,  0.901797  , -0.35757434, -0.30652952,
       -0.08591185,  0.72819257,  0.04963865, -0.264116  ,  0.5370738 ,
       -0.31950223, -0.3057032 , -0.18715814,  0.56429446,  0.12669842,
        0.322113  , -0.40736705, -0.44151682, -0.6654971 , -0.18048497,
       -0.0548893 , -0.15051961,  0.00946367, -0.15849464,  0.08875548,
        0.40782216,  0.4421426 , -0.3373687 , -0.12371846, -0.03128309,
        0.38177872,  0.00501736,  0.64645064,  0.4945376 , -1.2142695 ,
        0.12749472,  0.00529871,  0.15061134,  0.1683728 ,  0.16198564,
        0.21687442, -0.15088524, -0.54595375,  0.75176764, -0.04777783,
       -0.2184319 , -0.6056651 , -0.960831  ,  0.14346454,  0.3143191 ,
        0.79012436,  0.23687984,  0.20420447, -0.19882146,  0.12060179,
       -0.8229695 ,  0.93036145, -0.28153038,  0.5408102 , -0.71517795,
        0.67960125,  0.18667972,  0.50425535, -0.5533707 ,  0.64

In [ ]:
model.wv["vegata"]

KeyError: "Key 'vegata' not present"

In [ ]:
a = [0.0]*100
df_covid["centroid"] = [a]*df_covid.shape[0]

for index, row in df_covid.iterrows():
    abstract = row['processed_abstract']
    total_sim = 0
    words = abstract.split(" ")
    centroid = np.array([0.0]*100)
    for word in words:
        try:
            b = model[word]
        except:
            continue
        centroid = np.add(centroid, b)

    df_covid.at[index,'centroid'] = centroid.tolist()

#### Let us create our base function, which when given a query will retrieve and rank documents based on similarity.

In [ ]:
def rank_docs(model, query, df_covid, num) :

    cosine_list = []

    a = []
    query = query.split(" ")
    for q in query:
        try:
            a.append(model[q])
        except:
            continue

    for index, row in df_covid.iterrows():
        centroid = row['centroid']
        total_sim = 0
        for a_i in a:
            cos_sim = np.dot(a_i, centroid)/(np.linalg.norm(a_i)*np.linalg.norm(centroid))
            total_sim += cos_sim
        cosine_list.append((row['title'], row['doi'], total_sim))


    cosine_list.sort(key=lambda x:x[2], reverse=True) ## in Descending order

    papers_list = []
    for item in cosine_list[:num]:
        papers_list.append((item[0], item[1], item[2]))
    return papers_list

#### Method to use the base function and retrieve top matching documents

In [ ]:
def query(query, top_matches=10):
    model_to_use = model
    df_covid_to_use = df_covid
    return rank_docs(model_to_use, query, df_covid_to_use, top_matches)

In [ ]:
query('origin of corona virus')

[('From chaos to control – experiences of healthcare workers during the early phase of the COVID-19 pandemic: a focus group study',
  '10.1186/s12913-021-07248-9',
  0),
 ('The Long-Term Effects of Labor Market Entry in a Recession: Evidence from the Asian Financial Crisis()',
  '10.1016/j.labeco.2020.101926',
  0),
 ('Emerging evidence of myocardial injury in COVID-19: A path through the smoke',
  '10.7150/thno.50788',
  0),
 ('Qualitative Analysis of Fractal-Fractional order COVID-19 Mathematical Model with Case Study of Wuhan',
  '10.1016/j.aej.2020.09.020',
  0),
 ('Ambulatory Rehabilitation of Patients Hospitalized with SARS CoV‐2 Infections: Early Pandemic Experience in New York City and Boston',
  '10.1002/pmrj.12506',
  0),
 ('Crystallographic and electrophilic fragment screening of the SARS-CoV-2 main protease',
  '10.1101/2020.05.27.118117',
  0),
 ('Linking Technology to Address the Social and Medical Determinants of Health for Safe Medicines Use',
  '10.1097/pts.00000000000

## Diffrence between skipgram and cbow.


| Aspect                 | CBOW       | Skip-Gram |
| ---------------------- | ---------- | --------- |
| Rare words             | ❌ Weak     | ✅ Strong  |
| Frequent words         | ✅ Good     | ✅ Good    |
| Semantic relationships | ❌ Moderate | ✅ Better  |
| Small datasets         | ❌ Worse    | ✅ Better  |


The difference is modest but consistent

In [ ]:
import numpy as np

sentences = [
    ["i", "deposited", "money", "in", "the", "bank"],
    ["we", "sat", "by", "the", "river", "bank"]
]

# Build vocabulary
vocab = sorted(set(word for sent in sentences for word in sent))
print(vocab)

# Map each word to an index
word_to_index = {word: i for i, word in enumerate(vocab)}

# Function to create one-hot vector
def one_hot(word):
    vec = np.zeros(len(vocab))
    vec[word_to_index[word]] = 1
    return vec

['bank', 'by', 'deposited', 'i', 'in', 'money', 'river', 'sat', 'the', 'we']


In [ ]:
bank_vector_1 = one_hot("bank")  # from sentence 1
bank_vector_2 = one_hot("bank")  # from sentence 2

print(bank_vector_1)
print(bank_vector_2)

[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


| Word       | bank | by | deposited | i | in | money | river | sat | the | we |
|------------|------|----|-----------|---|----|-------|-------|-----|-----|----|
| bank       | 1    | 0  | 0         | 0 | 0  | 0     | 0     | 0   | 0   | 0  |
| by         | 0    | 1  | 0         | 0 | 0  | 0     | 0     | 0   | 0   | 0  |
| deposited  | 0    | 0  | 1         | 0 | 0  | 0     | 0     | 0   | 0   | 0  |
| i          | 0    | 0  | 0         | 1 | 0  | 0     | 0     | 0   | 0   | 0  |
| in         | 0    | 0  | 0         | 0 | 1  | 0     | 0     | 0   | 0   | 0  |
| money      | 0    | 0  | 0         | 0 | 0  | 1     | 0     | 0   | 0   | 0  |
| river      | 0    | 0  | 0         | 0 | 0  | 0     | 1     | 0   | 0   | 0  |
| sat        | 0    | 0  | 0         | 0 | 0  | 0     | 0     | 1   | 0   | 0  |
| the        | 0    | 0  | 0         | 0 | 0  | 0     | 0     | 0   | 1   | 0  |
| we         | 0    | 0  | 0         | 0 | 0  | 0     | 0     | 0   | 0   | 1  |